# Sense HAT API Tour with Java 25 and Pi4J

A hands-on tour of the entire `com.pi4j.drivers.hat.raspberry.SenseHat` API. Each section exercises one facet of the HAT so you can copy the individual snippets into your own projects.

Reference: [SenseHat.java](https://github.com/igfasouza/pi4j-drivers/blob/igfasouza/src/main/java/com/pi4j/drivers/hat/raspberry/SenseHat.java)

The tour covers:

1. Setup and lifecycle
2. Environmental sensors (temperature, humidity, pressure, light)
3. IMU (accelerometer, gyroscope, magnetometer, compass)
4. LED matrix pixels
5. LED matrix orientation (rotation, flip)
6. LED matrix text (letters, scrolling messages)
7. Underlying display driver access
8. Joystick / GameController (polling, blocking, listener)
9. `getAllSensors()` overview
10. Cleanup

Expected runtime environment:

- Raspberry Pi with a Sense HAT attached
- I2C enabled
- Docker container with `/dev/i2c-1`, `/dev/fb0`, `/dev/input/*` mapped through
- Java 25
- Jupyter Java (IJava) kernel


## 1. Maven Dependencies

The Sense HAT driver plus its Pi4J core and Linux FFM plugin.


In [ ]:
%maven com.pi4j:pi4j-core:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-plugin-ffm:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-drivers:1.1.0


## 2. Create a Pi4J Context and a SenseHat

`SenseHat` has two constructors:

- `SenseHat(Context)` — default orientation
- `SenseHat(Context, Rotation)` — start the LED matrix pre-rotated

`SenseHat` implements `AutoCloseable`, so you'd typically use it inside a try-with-resources block. In a notebook we keep it around across cells and close it in the last cell.


In [ ]:
import com.pi4j.Pi4J;
import com.pi4j.context.Context;
import com.pi4j.drivers.hat.raspberry.SenseHat;
import com.pi4j.drivers.display.Rotation;

Context pi4j = Pi4J.newAutoContext();

// Default orientation. To start rotated, use: new SenseHat(pi4j, Rotation.DEG_180)
SenseHat senseHat = new SenseHat(pi4j);

System.out.println("SenseHat ready");


## 3. Environmental Sensors

The Sense HAT ships with three environmental chips:

| Sensor  | Chip     | Reading                                        |
|---------|----------|------------------------------------------------|
| HTS221  | Humidity | `getHumidity()`, `getTemperatureFromHumidity()`|
| LPS25H  | Pressure | `getPressure()`, `getTemperatureFromPressure()`|
| TCS3400 | Light    | `getColour()` — returns `[r, g, b, c]`         |

Each sensor is also exposed as a raw driver via `getHumiditySensor()`, `getPressureSensor()`, `getLightSensor()` for advanced use.


In [ ]:
double humidity        = senseHat.getHumidity();
double tempFromHumidity = senseHat.getTemperatureFromHumidity();

double pressure        = senseHat.getPressure();
double tempFromPressure = senseHat.getTemperatureFromPressure();

double temperature = (tempFromHumidity + tempFromPressure) / 2.0;

double[] colour = senseHat.getColour();

System.out.printf("Humidity:              %.2f %%RH%n", humidity);
System.out.printf("Temperature (HTS221):  %.2f C%n", tempFromHumidity);
System.out.printf("Temperature (LPS25H):  %.2f C%n", tempFromPressure);
System.out.printf("Temperature (avg):     %.2f C%n", temperature);
System.out.printf("Pressure:              %.2f hPa%n", pressure);
System.out.printf("Colour [R,G,B,Clear]:  [%.0f, %.0f, %.0f, %.0f]%n",
    colour[0], colour[1], colour[2], colour[3]);


## 4. IMU — Accelerometer, Gyroscope, Magnetometer

The LSM9DS1 chip on the Sense HAT is a 9-DoF IMU. The driver exposes:

- Accelerometer: `getAccelerometerRaw()` — [x, y, z] in g
- Gyroscope: `getGyroscopeRaw()` — [x, y, z] in deg/s
- Fused orientation: `getOrientationDegrees()`, `getOrientationRadians()` — [pitch, roll, yaw]
- Magnetometer: `getCompassRaw()` — [x, y, z] raw magnetic field
- Compass heading: `getCompass()` — heading in degrees

`setImuConfig(compass, gyro, accel)` lets you selectively enable subsystems (turning off unused ones saves power and CPU).


In [ ]:
senseHat.setImuConfig(true, true, true);

double[] accel      = senseHat.getAccelerometerRaw();
double[] gyro       = senseHat.getGyroscopeRaw();
double[] orientDeg  = senseHat.getOrientationDegrees();
double[] orientRad  = senseHat.getOrientationRadians();
double[] mag        = senseHat.getCompassRaw();
double   heading    = senseHat.getCompass();

System.out.printf("Accel  [g]     : x=%.3f y=%.3f z=%.3f%n", accel[0], accel[1], accel[2]);
System.out.printf("Gyro   [deg/s] : x=%.3f y=%.3f z=%.3f%n", gyro[0],  gyro[1],  gyro[2]);
System.out.printf("Orient [deg]   : pitch=%.2f roll=%.2f yaw=%.2f%n",
    orientDeg[0], orientDeg[1], orientDeg[2]);
System.out.printf("Orient [rad]   : pitch=%.3f roll=%.3f yaw=%.3f%n",
    orientRad[0], orientRad[1], orientRad[2]);
System.out.printf("Mag raw        : x=%.1f y=%.1f z=%.1f%n", mag[0], mag[1], mag[2]);
System.out.printf("Compass        : %.1f deg%n", heading);


## 5. LED Matrix — Pixel Operations

The 8x8 LED matrix is addressed as either:

- Packed 24-bit RGB `int` — `0xRRGGBB`
- Separate `r, g, b` components (0–255)

Methods:

- `clear()` — turn all LEDs off
- `fill(int color)` / `fill(int r, int g, int b)` — fill the whole matrix
- `setPixel(x, y, color)` / `setPixel(x, y, r, g, b)`
- `getPixel(x, y)` — packed RGB int
- `getPixels()` / `setPixels(int[])` / `setPixels(int[][])`

Coordinates: `x` = 0..7 left-to-right, `y` = 0..7 top-to-bottom.


In [ ]:
final int RED   = 0xFF0000;
final int GREEN = 0x00FF00;
final int BLUE  = 0x0000FF;
final int BLACK = 0x000000;

senseHat.clear();
Thread.sleep(300);

senseHat.fill(RED);
Thread.sleep(500);

senseHat.fill(0, 255, 0);
Thread.sleep(500);

// Draw a diagonal line in blue, one pixel at a time.
senseHat.clear();
for (int i = 0; i < 8; i++) {
    senseHat.setPixel(i, i, BLUE);
    Thread.sleep(60);
}
Thread.sleep(500);

// Read back the pixel we set.
int pixel = senseHat.getPixel(3, 3);
System.out.printf("Pixel (3,3) = 0x%06X%n", pixel & 0xFFFFFF);


### 5.1. Bulk pixel updates with `setPixels`

Push an entire 8x8 buffer in one call — much faster than 64 `setPixel` calls. Supports both a 1-D flat `int[64]` and a 2-D `int[8][8]`.


In [ ]:
int O = 0x000000; // off
int Y = 0xFFFF00; // yellow

// A tiny smiley face — 8x8, row-major.
int[][] smiley = {
    { O, O, Y, Y, Y, Y, O, O },
    { O, Y, Y, Y, Y, Y, Y, O },
    { Y, Y, O, Y, Y, O, Y, Y },
    { Y, Y, Y, Y, Y, Y, Y, Y },
    { Y, O, Y, Y, Y, Y, O, Y },
    { Y, Y, O, O, O, O, Y, Y },
    { O, Y, Y, Y, Y, Y, Y, O },
    { O, O, Y, Y, Y, Y, O, O },
};

senseHat.setPixels(smiley);
Thread.sleep(1500);

// Same buffer as a flat array — paint a rainbow gradient.
int[] flat = new int[64];
for (int i = 0; i < 64; i++) {
    float hue = i / 64.0f;
    flat[i] = java.awt.Color.HSBtoRGB(hue, 1.0f, 0.4f) & 0xFFFFFF;
}
senseHat.setPixels(flat);
Thread.sleep(1500);

// Snapshot the whole matrix.
int[] snapshot = senseHat.getPixels();
System.out.println("Snapshot length: " + snapshot.length);
System.out.printf("Top-left pixel:  0x%06X%n", snapshot[0] & 0xFFFFFF);


## 6. LED Matrix — Orientation

The driver can rotate or flip the drawing surface. This changes only how future draw calls map to physical LEDs — it does not touch the pixel buffer.

- `setRotation(Rotation)` where `Rotation` is one of `DEG_0`, `DEG_90`, `DEG_180`, `DEG_270`
- `flipHorizontal()`
- `flipVertical()`


In [ ]:
// Draw an arrow that points "up" in the current orientation.
int Y2 = 0xFFAA00; // amber
int O2 = 0x000000;
int[][] arrow = {
    { O2, O2, O2, Y2, Y2, O2, O2, O2 },
    { O2, O2, Y2, Y2, Y2, Y2, O2, O2 },
    { O2, Y2, Y2, Y2, Y2, Y2, Y2, O2 },
    { Y2, Y2, O2, Y2, Y2, O2, Y2, Y2 },
    { O2, O2, O2, Y2, Y2, O2, O2, O2 },
    { O2, O2, O2, Y2, Y2, O2, O2, O2 },
    { O2, O2, O2, Y2, Y2, O2, O2, O2 },
    { O2, O2, O2, Y2, Y2, O2, O2, O2 },
};

for (Rotation r : Rotation.values()) {
    senseHat.setRotation(r);
    senseHat.setPixels(arrow);
    System.out.println("Rotation: " + r);
    Thread.sleep(800);
}

// Reset and try flips.
senseHat.setRotation(Rotation.DEG_0);
senseHat.setPixels(arrow);
Thread.sleep(600);

senseHat.flipHorizontal();
System.out.println("Flipped horizontally");
Thread.sleep(800);

senseHat.flipVertical();
System.out.println("Flipped vertically");
Thread.sleep(800);

senseHat.clear();


## 7. LED Matrix — Text

The `showLetter(...)` family renders a single character. `showMessage(...)` scrolls a whole string across the matrix.

- `showLetter(char)` — default colours
- `showLetter(char, textColor)`
- `showLetter(char, textColor, backColor)`
- `showMessage(text)`
- `showMessage(text, scrollSpeedMillis)`
- `showMessage(text, scrollSpeedMillis, textColor)`
- `showMessage(text, scrollSpeedMillis, textColor, backColor)`
- `showMessage(text, scrollSpeedMillis, textColor, backColor, ScrollDirection)`

`scrollSpeedMillis` is the delay per column, so smaller = faster.


In [ ]:
import com.pi4j.drivers.hat.raspberry.ScrollDirection;

// Single letters with different colour combinations.
senseHat.showLetter('J');
Thread.sleep(700);

senseHat.showLetter('A', 0x00FFFF);            // cyan on default background
Thread.sleep(700);

senseHat.showLetter('V', 0xFFFFFF, 0x000044);  // white on dark blue
Thread.sleep(700);

// Scrolling messages.
senseHat.showMessage("Hi");
senseHat.showMessage("Sense HAT", 60);
senseHat.showMessage("Pi4J", 60, 0xFF00FF);
senseHat.showMessage("Java 25", 60, 0xFFFFFF, 0x001133);
senseHat.showMessage("<--", 60, 0x00FF00, 0x000000, ScrollDirection.RIGHT_TO_LEFT);

senseHat.clear();


## 8. Direct access to the display driver

For advanced use (custom graphics, external animation loops), the underlying driver is exposed as:

- `getDisplayDriver()` — the raw `GraphicsDisplayDriver`
- `getDisplay()` — a `GraphicsDisplay` wrapper (higher-level, supports drawing primitives)


In [ ]:
var driver = senseHat.getDisplayDriver();
var display = senseHat.getDisplay();

System.out.println("Display driver class: " + driver.getClass().getName());
System.out.println("Display class:        " + display.getClass().getName());


## 9. Joystick

The 5-way joystick is exposed as a `GameController`. Three access patterns:

- **Polling** — `getEvents()` returns any events since the last call, non-blocking.
- **Blocking** — `waitForEvent()` blocks until the next event.
- **Listener** — `addJoystickListener(Consumer<Event>)` fires callbacks on a background thread; remove with `removeJoystickListener(...)`.

Each `Event` is `record Event(Instant timestamp, GameController.Key key, Action action)`, where `Action` is `PRESSED`, `RELEASED`, or `HELD`.

You can also grab the raw input driver via `getInputDriver()`.


### 9.1. Polling pattern

Poll for a few seconds and print anything the user does on the joystick.


In [ ]:
import com.pi4j.drivers.hat.raspberry.SenseHat.Event;

long deadline = System.currentTimeMillis() + 5_000;

System.out.println("Move the joystick for 5 seconds...");

while (System.currentTimeMillis() < deadline) {
    for (Event e : senseHat.getEvents()) {
        System.out.printf("%s  key=%s  action=%s%n",
            e.timestamp(), e.key(), e.action());
    }
    Thread.sleep(50);
}

System.out.println("Done polling.");


### 9.2. Blocking pattern

`waitForEvent()` returns the next event. Great for simple state machines.


In [ ]:
System.out.println("Press the joystick to continue...");
Event event = senseHat.waitForEvent();
System.out.printf("Got: key=%s action=%s%n", event.key(), event.action());


### 9.3. Listener pattern

Register a callback that fires on the joystick's own thread. The callback below lights a pixel in the direction of the last press. Runs until you press the middle key.


In [ ]:
import com.pi4j.io.gpio.digital.GameController;
import com.pi4j.drivers.hat.raspberry.SenseHat.Action;
import java.util.concurrent.CountDownLatch;
import java.util.function.Consumer;

CountDownLatch done = new CountDownLatch(1);

Consumer<Event> listener = e -> {
    if (e.action() != Action.PRESSED) return;

    senseHat.clear();
    GameController.Key k = e.key();

    if (k == GameController.Key.UP)         senseHat.setPixel(4, 0, 0x00FF00);
    else if (k == GameController.Key.DOWN)  senseHat.setPixel(4, 7, 0x00FF00);
    else if (k == GameController.Key.LEFT)  senseHat.setPixel(0, 4, 0x00FF00);
    else if (k == GameController.Key.RIGHT) senseHat.setPixel(7, 4, 0x00FF00);
    else if (k == GameController.Key.START) {
        senseHat.showLetter('.', 0xFF0000);
        done.countDown();
    }
};

senseHat.addJoystickListener(listener);

System.out.println("Move the joystick. Press CENTER (START) to end.");
done.await();

senseHat.removeJoystickListener(listener);
senseHat.clear();
System.out.println("Listener removed.");


## 10. `getAllSensors()`

Returns a list of every high-level `Sensor` the driver exposes — useful for generic dashboards or logging pipelines that don't want to hardcode each chip.


In [ ]:
senseHat.getAllSensors().forEach(sensor ->
    System.out.println(sensor.getClass().getSimpleName() + " -> " + sensor));


## 11. Cleanup

`SenseHat` implements `AutoCloseable`. Call `close()` to release the LED matrix and joystick handles, then shut down the Pi4J context. Errors during shutdown are logged but not rethrown so the notebook still finishes.


In [ ]:
try {
    senseHat.clear();
    senseHat.close();
} catch (Exception e) {
    System.out.println("SenseHat close warning: " + e.getMessage());
}

try {
    pi4j.shutdown();
} catch (Exception e) {
    System.out.println("Pi4J shutdown warning: " + e.getMessage());
}

System.out.println("Done.");
